# OpenNext Layout Planner — Colab **API server**
Serves the trained Qwen pipeline as a public HTTP API. POST a prompt to `/generate`, get the model output back (Stage 1 LayoutPlan + optional Stage 2 filled page).

**Runtime > Change runtime type > T4 GPU** first. Run cells 1-7 in order; cell 7 prints your public URL.

In [ ]:
# 1) Install deps (Colab already has CUDA torch)
!pip -q install "transformers>=4.44" "peft>=0.11" accelerate safetensors requests json-repair fastapi "uvicorn[standard]" nest_asyncio
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - Runtime > Change runtime type > T4 GPU")

In [ ]:
# 2) Upload the LoRA adapter zip (qwen-layout-planner.zip)
import os, glob, zipfile
from google.colab import files
up = files.upload()                      # choose qwen-layout-planner.zip
with zipfile.ZipFile(next(iter(up))) as z: z.extractall("/content/_adapter")
ADAPTER_DIR = os.path.dirname(glob.glob("/content/_adapter/**/adapter_config.json", recursive=True)[0])
print("ADAPTER_DIR =", ADAPTER_DIR)
# Or from HF Hub instead:  ADAPTER_DIR = "your-username/qwen-layout-planner"

In [ ]:
# 3) Config — SYSTEM byte-identical to inference.py; ALIASES/FILLER match pipeline.py
import json, re, os

BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"
INPUT_PREFIX = "plan layout: "

SYSTEM = (
    "You are a website STRUCTURE planner for OpenNext CMS. "
    "Given a brief, return ONE minified JSON LayoutPlan and nothing else. "
    "Shape: {meta:{business,style,websiteType,pageName,location,complexity},"
    "page:{theme:{primaryColor,secondaryColor,backgroundColor,textColor,fontFamily,borderRadius},"
    "blocks:[{id,type,order,styles,content}]}}. "
    "Rules: blocks[0] is a navigation bar and the last block is a footer (exactly one each); "
    "exactly one hero, placed before any feature/testimonial/pricing/content/stats section; "
    "every block.content is {} (NO copy); ids are unique kebab-case; order is 0..n-1; "
    "colours are hex and section backgrounds vary; "
    "complexity simple=4-6 blocks, standard=7-10, rich=11-15."
)

ALIASES = {
    "nav-bar": "navigation", "hero": "hero", "hero-main": "heroCentered", "hero-centered": "heroCentered",
    "feature-trio": "featureGrid", "feature-vertical": "featureCards", "feature-horizontal": "featureCards",
    "feature-boxed": "featureCards", "feature-zigzag": "featureCards", "feature-side-image": "featureCards",
    "feature-checklist": "featureGrid", "feature-list": "featureGrid", "content-features": "featureGrid",
    "content-gallery": "gallery", "content-split": "featureCards", "content-trio": "featureGrid",
    "content-icons": "featureGrid", "content-categories": "gallery", "content-detail": "featureCards",
    "testimonial-main": "testimonials", "testimonial-single": "testimonials",
    "testimonial-single-large": "testimonials", "stats": "statistics", "statistics-main": "statistics",
    "statistics-side-image": "statistics", "statistics-boxed": "statistics", "ecommerce-grid": "gallery",
    "ecommerce-detail": "gallery", "pricing": "pricing", "cta": "cta", "contact": "contact",
    "contact-simple": "contact", "footer": "footer",
}

FILLER_SYSTEM = (
    "You are the Stage-2 CONTENT filler for OpenNext CMS. You are given a FIXED page "
    "structure: a list of blocks, each with an id, a type, a styles object, and an empty "
    "data object. Fill ONLY the copy.\n\n"
    "RULES:\n"
    "- Return ONE JSON object: {\"meta\":{...},\"theme\":{...},\"blocks\":[{id,type,styles,data}]}.\n"
    "- Keep EVERY block's id, type, styles and ORDER exactly as given. Do NOT add, remove, "
    "reorder, or retype blocks.\n"
    "- Populate each block's `data` with realistic, niche-specific copy for the brief "
    "(headlines <=10 words, subheadlines <=30 words, feature text <=20 words). No lorem ipsum.\n"
    "- Use the data shape appropriate to each v2 type: navigation{logo,links,cta}, "
    "heroCentered/hero{eyebrow,headline,subheadline,cta_primary,media}, "
    "featureGrid/featureCards{headline,features|cards:[{title,description,icon}]}, "
    "testimonials{headline,items:[{quote,author_name,author_title}]}, "
    "statistics{headline,stats:[{value,label}]}, pricing{headline,plans:[{name,price,features}]}, "
    "cta{headline,subheadline,cta_primary}, contact{headline,email,phone,address}, "
    "gallery{headline,items:[{src,caption}]}, footer{logo,columns:[{title,links}],copyright}.\n"
    "- Every media/avatar/logo field gets a real Unsplash/Pexels URL matching the niche.\n"
    "Return JSON only - no markdown, no prose."
)

def parse_plan_json(raw):
    if not raw: return None
    s = raw.strip()
    m = re.search(r"```(?:json)?\s*(.*?)```", s, re.S)
    if m: s = m.group(1).strip()
    try: return json.loads(s)
    except Exception: pass
    i, j = s.find("{"), s.rfind("}")
    if i != -1 and j != -1 and j > i:
        frag = s[i:j+1]
        try: return json.loads(frag)
        except Exception:
            try:
                from json_repair import repair_json
                return json.loads(repair_json(frag))
            except Exception: return None
    return None

NAV, HERO, FOOT = {"nav-bar"}, {"hero", "hero-main", "hero-centered"}, {"footer"}
def validate(plan):
    if not isinstance(plan, dict): return False
    page = plan.get("page")
    if not isinstance(page, dict) or not isinstance(page.get("blocks"), list) or not page["blocks"]: return False
    t = [b.get("type") for b in page["blocks"] if isinstance(b, dict)]
    return t[0] in NAV and t[-1] in FOOT and sum(x in HERO for x in t) == 1

print("config loaded")

In [ ]:
# 4) Load base Qwen + LoRA adapter on the GPU (fp16)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tok_src = ADAPTER_DIR if os.path.exists(os.path.join(ADAPTER_DIR, "tokenizer_config.json")) else BASE_MODEL
tok = AutoTokenizer.from_pretrained(tok_src)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto"),
    ADAPTER_DIR).eval()
print("model ready on", model.device)

In [ ]:
# 5) Pipeline functions
import requests

def generate(brief, max_new_tokens=768, do_sample=False, temperature=1.0):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": INPUT_PREFIX + brief.strip()}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(model.device)
    kw = dict(max_new_tokens=max_new_tokens, do_sample=do_sample, pad_token_id=tok.pad_token_id)
    if do_sample: kw["temperature"] = temperature
    with torch.no_grad():
        out = model.generate(**enc, **kw)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

def stage1(brief):
    plan = parse_plan_json(generate(brief, do_sample=False))
    if plan and validate(plan): return plan, "model"
    plan2 = parse_plan_json(generate(brief, do_sample=True, temperature=0.7))
    if plan2 and validate(plan2): return plan2, "model-retry"
    return (plan or plan2), "model-unvalidated"

def to_v2_skeleton(plan):
    page = plan.get("page", {}); meta = plan.get("meta", {}); th = page.get("theme", {})
    blocks = [{"id": b.get("id"), "type": ALIASES.get(b.get("type"), b.get("type")),
               "styles": b.get("styles", {}), "data": {}} for b in page.get("blocks", [])]
    return {"meta": {k: meta.get(k) for k in ("business", "websiteType", "style", "pageName", "location")},
            "theme": {"primary_color": th.get("primaryColor"), "secondary_color": th.get("secondaryColor"),
                      "background_color": th.get("backgroundColor"), "text_color": th.get("textColor"),
                      "font_body": th.get("fontFamily"), "border_radius": th.get("borderRadius")},
            "blocks": blocks}

def fill(skeleton, brief, model_id="openai/gpt-4o-mini"):
    msgs = [{"role": "system", "content": FILLER_SYSTEM},
            {"role": "user", "content": f"BRIEF: {brief}\n\nSTRUCTURE (fill each block's data, keep everything else):\n"
                                        + json.dumps(skeleton, ensure_ascii=False)}]
    r = requests.post("https://openrouter.ai/api/v1/chat/completions",
                      headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
                               "Content-Type": "application/json"},
                      json={"model": model_id, "temperature": 0.7, "max_tokens": 4000, "messages": msgs},
                      timeout=120)
    r.raise_for_status()
    return parse_plan_json(r.json()["choices"][0]["message"]["content"]) or skeleton

print("functions ready")

In [ ]:
# 6) Define the FastAPI app
import getpass, time
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# Optional: set an OpenRouter key to enable Stage-2 copy fill. Leave blank -> Stage 1 only.
if not os.environ.get("OPENROUTER_API_KEY"):
    try:
        os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key (blank = Stage 1 only): ")
    except Exception:
        os.environ["OPENROUTER_API_KEY"] = ""

FILL_MODEL = os.environ.get("OPENROUTER_MODEL", "openai/gpt-4o-mini")

app = FastAPI(title="OpenNext Layout Planner API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class GenReq(BaseModel):
    prompt: str
    stage2: bool = True          # also run the copy fill (needs OpenRouter key)

@app.get("/")
def health():
    return {"status": "ok", "model": BASE_MODEL, "device": str(model.device),
            "stage2_available": bool(os.environ.get("OPENROUTER_API_KEY", "").strip())}

@app.post("/generate")
def generate_endpoint(req: GenReq):
    t0 = time.time()
    plan, source = stage1(req.prompt)
    out = {"brief": req.prompt, "stage1_source": source,
           "stage1_layout_plan": plan, "stage1_seconds": round(time.time() - t0, 1)}
    if req.stage2 and os.environ.get("OPENROUTER_API_KEY", "").strip() and plan:
        t1 = time.time()
        out["stage2_filled_page"] = fill(to_v2_skeleton(plan), req.prompt, FILL_MODEL)
        out["stage2_seconds"] = round(time.time() - t1, 1)
    return out

print("FastAPI app defined  (endpoints: GET /  and  POST /generate)")

In [ ]:
# 7) Launch the server + a PUBLIC url (cloudflared quick tunnel, no signup)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

import nest_asyncio, uvicorn, threading, subprocess, re, time
nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
                 daemon=True).start()
time.sleep(4)

with open("cf.log", "w") as f:
    subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                     stdout=f, stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(40):
    time.sleep(1)
    try: txt = open("cf.log").read()
    except FileNotFoundError: txt = ""
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", txt)
    if m: PUBLIC_URL = m.group(0); break

print("\n" + "=" * 64)
print("PUBLIC API URL:", PUBLIC_URL or "NOT READY - re-run this cell / check cf.log")
print("Health : GET  ", (PUBLIC_URL or "<url>") + "/")
print("Generate: POST", (PUBLIC_URL or "<url>") + "/generate   body {\"prompt\":\"...\",\"stage2\":true}")
print("=" * 64)
print("Note: trycloudflare URLs are temporary and change every run. Keep this cell running to keep the API up.")

In [ ]:
# 8) Test it (call your own API)
import requests, json
resp = requests.post(PUBLIC_URL + "/generate",
                     json={"prompt": "Create a modern ecommerce store for handmade leather goods.",
                           "stage2": True}, timeout=300)
print("status:", resp.status_code)
print(json.dumps(resp.json(), indent=2, ensure_ascii=False)[:2500])